In [3]:
import numpy as np

def rss(y):
    y_mean = np.mean(y)
    total = 0
    for i in range(y.size):
        total += (y[i] - y_mean) ** 2
    return total

In [5]:
def rss_reduction(y_parent, y_left, y_right):
    return rss(y_parent) - rss(y_left) - rss(y_right)

In [6]:
def create_thresholds(x):
    x = np.sort(np.unique(x))
    y = []
    for i in range(len(x)-1):
        y.append((x[i] + x[i+1])/2)
    return y

def split(x, y, threshold):
    y_left = []
    y_right = []
    for i in range(len(y)):
        if(x[i] <= threshold):
            y_left.append(y[i])
        else:
            y_right.append(y[i])
    return np.array(y_left), np.array(y_right)

def best_split_for_feature_regression(x, y):
    thresholds = create_thresholds(x)

    best_threshold = None
    best_reduction = -np.inf

    for threshold in thresholds:
        y_left, y_right = split(x, y, threshold)
        rss_val = rss_reduction(y,y_left,y_right)
        if(rss_val > best_reduction):
            best_reduction = rss_val
            best_threshold = threshold
        
    return best_threshold, best_reduction

In [13]:
def best_split_regression(X, y):
    best_feature = None
    best_threshold = None
    best_reduction = -np.inf

    for j in range(X.shape[1]):
        threshold,reduction = best_split_for_feature_regression(X[:, j], y)
        if(reduction > best_reduction):
            best_feature = j
            best_threshold = threshold
            best_reduction = reduction
        
    return best_feature, best_threshold, best_reduction

In [15]:
def split_data(X, y, feature, threshold):
    left_mask = X[:, feature] <= threshold
    right_mask = X[:, feature] > threshold

    X_left = X[left_mask]
    y_left = y[left_mask]

    X_right = X[right_mask]
    y_right = y[right_mask]

    return X_left, y_left, X_right, y_right

def build_regression_tree(X, y,depth=0,max_depth=3,min_samples_split=2):
    if(np.all(y == y[0])):
        return {"leaf": True, "prediction" : float(y[0])}
    if(depth >= max_depth or len(y) < min_samples_split):
        return {"leaf": True, "prediction" : np.mean(y)}

    feature, threshold, reduction = best_split_regression(X, y)
    if reduction <= 0:
        return {"leaf": True, "prediction": np.mean(y)}
    
    X_left, y_left, X_right, y_right = split_data(X, y, feature, threshold)
    left_tree = build_regression_tree(X_left,y_left,depth+1,max_depth,min_samples_split)
    right_tree = build_regression_tree(X_right,y_right,depth+1,max_depth,min_samples_split)

    return {
        "leaf": False,
        "feature": feature,
        "threshold": threshold,
        "left": left_tree,
        "right": right_tree
    }

In [10]:
def predict_one_regression(x, tree):
    if tree["leaf"]:
        return tree["prediction"]

    if x[tree["feature"]] <= tree["threshold"]:
        return predict_one_regression(x, tree["left"])
    else:
        return predict_one_regression(x, tree["right"])

In [11]:
def predict_regression(X, tree):
    predictions = []

    for row in X:
        predictions.append(predict_one_regression(row,tree))

    return np.array(predictions)

In [16]:
X = np.array([
    [1],
    [2],
    [3],
    [8],
    [9],
    [10]
])

y = np.array([2.0, 2.5, 3.0, 8.0, 8.5, 9.0])

tree = build_regression_tree(X, y, max_depth=2)

print(tree)
print(predict_regression(X, tree))
print(y)

{'leaf': False, 'feature': 0, 'threshold': np.float64(5.5), 'left': {'leaf': False, 'feature': 0, 'threshold': np.float64(1.5), 'left': {'leaf': True, 'prediction': 2.0}, 'right': {'leaf': True, 'prediction': np.float64(2.75)}}, 'right': {'leaf': False, 'feature': 0, 'threshold': np.float64(8.5), 'left': {'leaf': True, 'prediction': 8.0}, 'right': {'leaf': True, 'prediction': np.float64(8.75)}}}
[2.   2.75 2.75 8.   8.75 8.75]
[2.  2.5 3.  8.  8.5 9. ]


In [21]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../../data")

df = pd.read_csv(DATA_DIR / "duolingo_flagship_v5.csv")
split_users = pd.read_csv(DATA_DIR / "split_users.csv")

print(df.columns.tolist())
print(split_users.columns.tolist())
print(split_users.head())

['practice_time', 'user_id', 'ui_language', 'learning_language', 'surface_form', 'lemma', 'pos', 'grammar_tags', 'lag_days', 'history_seen', 'history_correct', 'session_seen', 'session_correct', 'p_recall', 'lexeme_id', 'lag_days_log', 'history_accuracy', 'difficulty_rank_in_language']
['user_id', 'split']
  user_id split
0   u:0Bk  test
1   u:27Z  test
2   u:F-1  test
3   u:FHQ  test
4   u:QMy  test


In [22]:
print(split_users["split"].value_counts())

split
cv      2125
test     375
Name: count, dtype: int64


In [23]:
cv_users = split_users.loc[split_users["split"] == "cv", "user_id"]
test_users = split_users.loc[split_users["split"] == "test", "user_id"]

cv_df = df[df["user_id"].isin(cv_users)].copy()
holdout_df = df[df["user_id"].isin(test_users)].copy()

print("CV rows:", len(cv_df))
print("CV users:", cv_df["user_id"].nunique())

print("Holdout rows:", len(holdout_df))
print("Holdout users:", holdout_df["user_id"].nunique())

print("User overlap:",len(set(cv_df["user_id"]) & set(holdout_df["user_id"])))

CV rows: 14438
CV users: 2125
Holdout rows: 1944
Holdout users: 375
User overlap: 0


In [25]:
features = [
    "lag_days",
    "history_seen",
    "history_correct",
    "history_accuracy",
    "lag_days_log",
]

target = "p_recall"

In [26]:
from sklearn.model_selection import GroupKFold
from sklearn.tree import DecisionTreeRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
import numpy as np

X = cv_df[features]
y = cv_df[target]
groups = cv_df["user_id"]

gkf = GroupKFold(n_splits=5)

fold_rmses = []

for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups), start=1):
    X_train = X.iloc[train_idx]
    y_train = y.iloc[train_idx]

    X_val = X.iloc[val_idx]
    y_val = y.iloc[val_idx]

    model = Pipeline([("tree", DecisionTreeRegressor(random_state=42))])
    model.fit(X_train, y_train)
    pred = model.predict(X_val)

    rmse = np.sqrt(mean_squared_error(y_val, pred))
    fold_rmses.append(rmse)

    print(f"Fold {fold}: {rmse:.6f}")

print()
print(f"Mean RMSE: {np.mean(fold_rmses):.6f}")
print(f"Std RMSE:  {np.std(fold_rmses):.6f}")

Fold 1: 0.370282
Fold 2: 0.375427
Fold 3: 0.363491
Fold 4: 0.360687
Fold 5: 0.378602

Mean RMSE: 0.369698
Std RMSE:  0.006814


### Untuned Decision Tree

5-fold GroupKFold CV RMSE:

- Mean RMSE: 0.369698
- Std RMSE: 0.006814

The unconstrained decision tree performs substantially worse than Ridge
(~0.2735 RMSE). Its flexibility allows it to fit noisy local patterns in the
training data, which generalize poorly across unseen users.

This does not show that tree models are inherently unsuitable; it shows that
tree complexity must be controlled.